# MediGuide AI — Kaggle QLoRA Fine-Tuning

آموزش آداپتر LoRA برای Qwen3-4B با دیتاست‌های MedQA و PubMedQA.

**مهم:** این پروژه آموزشی/پژوهشی است؛ برای تشخیص، تجویز یا تصمیم‌گیری بالینی مناسب نیست. پیش از هر استفاده واقعی پزشکی، ارزیابی ایمنی و بازبینی متخصص لازم است.

در Kaggle گزینه GPU T4 x2 و Internet را روشن کن. این نوت‌بوک فعلاً مدل را روی GPU شماره 0 اجرا می‌کند؛ دو T4 حافظه مشترک ندارند.

## 1. بررسی GPU

In [ ]:
import os, sys, gc, json, random, inspect, warnings
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
import torch
print(sys.version, torch.__version__)
assert torch.cuda.is_available(), "GPU فعال نیست؛ در Kaggle از Settings گزینه GPU را روشن کن."
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

## 2. نصب کتابخانه‌ها
PyTorch پیش‌فرض Kaggle را دوباره نصب نمی‌کنیم.

In [ ]:
%pip install -q -U transformers datasets accelerate peft bitsandbytes sentencepiece scikit-learn

## 3. Import و نسخه‌ها
اگر import بعد از نصب خطا داد، Restart Session کن و از ابتدا اجرا کن.

In [ ]:
import transformers, datasets, peft, bitsandbytes
from datasets import load_dataset, concatenate_datasets, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("peft", peft.__version__)
print("bitsandbytes", bitsandbytes.__version__)

## 4. دریافت دیتاست‌ها

In [ ]:
medqa = load_dataset("GBaker/MedQA-USMLE-4-options")
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled")
print("MedQA:", medqa)
print("PubMedQA:", pubmedqa)
print("MedQA columns:", medqa["train"].column_names)
print("PubMedQA columns:", pubmedqa["train"].column_names)
print("MedQA sample:", medqa["train"][0])
print("PubMedQA sample:", pubmedqa["train"][0])

## 5. تبدیل داده‌ها به instruction

In [ ]:
def txt(x):
    if x is None: return ""
    if isinstance(x, (list, tuple)): return "\n".join(map(str, x))
    return str(x)

def format_medqa(x):
    required = {"question", "options", "answer"}
    if not required.issubset(x):
        raise KeyError(f"MedQA columns mismatch. Missing: {required-set(x)}; found: {list(x)}")
    opts = x["options"]
    if isinstance(opts, dict):
        choices = "\n".join(f"{k}: {v}" for k, v in opts.items())
    elif isinstance(opts, list):
        choices = "\n".join(f"{i+1}: {v}" for i, v in enumerate(opts))
    else:
        choices = txt(opts)
    q, ans = txt(x["question"]), txt(x["answer"])
    if not q.strip() or not ans.strip(): return None
    extra = txt(x.get("explanation", ""))
    if x.get("answer_idx") is not None: extra = f"Answer index: {txt(x['answer_idx'])}\n" + extra
    out = f"Correct answer: {ans}"
    if extra.strip(): out += f"\nExplanation: {extra}"
    return {"instruction":"Answer this medical education question. Select the correct option and explain briefly.", "input":f"Question: {q}\nOptions:\n{choices}", "output":out, "source":"MedQA"}

def format_pubmedqa(x):
    required = {"question", "context", "final_decision", "long_answer"}
    if not required.issubset(x):
        raise KeyError(f"PubMedQA columns mismatch. Missing: {required-set(x)}; found: {list(x)}")
    ctx = x["context"]
    if isinstance(ctx, dict): ctx = txt(ctx.get("contexts", []))
    else: ctx = txt(ctx)
    q, decision = txt(x["question"]), txt(x["final_decision"])
    if not q.strip() or not decision.strip(): return None
    return {"instruction":"Answer the medical research question using the supplied scientific context. State uncertainty when evidence is insufficient.", "input":f"Question: {q}\nScientific context:\n{ctx}", "output":f"Answer: {decision}\nExplanation: {txt(x['long_answer'])}", "source":"PubMedQA"}

med_rows = [format_medqa(x) for x in medqa["train"]]
pub_rows = [format_pubmedqa(x) for x in pubmedqa["train"]]
med_rows = [x for x in med_rows if x is not None]
pub_rows = [x for x in pub_rows if x is not None]
med_ds, pub_ds = Dataset.from_list(med_rows), Dataset.from_list(pub_rows)
print("Formatted rows:", len(med_ds), len(pub_ds))
print(med_ds[0])

## 6. ترکیب و تقسیم داده
Validation از بخش train ساخته می‌شود؛ این نتیجه آزمون رسمی یا اعتبارسنجی بالینی نیست.

In [ ]:
all_data = concatenate_datasets([med_ds, pub_ds]).shuffle(seed=SEED)
parts = all_data.train_test_split(test_size=0.08, seed=SEED)
train_dataset, eval_dataset = parts["train"], parts["test"]
print("Train:", len(train_dataset), "Validation:", len(eval_dataset))

## 7. بارگذاری Qwen3-4B با 4-bit

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B"
MAX_LENGTH = 1024
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.truncation_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    trust_remote_code=True
)
model.config.use_cache = False
print("Model loaded on:", model.get_input_embeddings().weight.device)

## 8. آماده‌سازی LoRA

In [ ]:
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 9. توکن‌سازی و تست یک نمونه
این روش خروجی `apply_chat_template` را ابتدا به متن تبدیل می‌کند تا مشکل نوع `Encoding` و دیکشنری پیش نیاید.

In [ ]:
SYSTEM_PROMPT = (
    "You are MediGuide AI, an educational medical information assistant. "
    "Provide clear, evidence-aware medical information. Do not diagnose individuals "
    "or prescribe medication. Never replace a clinician. State uncertainty when evidence "
    "is insufficient. For emergency symptoms, advise immediate emergency medical help."
)

def chat_text(messages, add_generation_prompt):
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=add_generation_prompt
    )

def ids(text):
    return [int(v) for v in tokenizer(text, add_special_tokens=False)["input_ids"]]

def tokenize_example(x):
    user_text = (
        f"Instruction:\n{x['instruction']}\n\n"
        f"Medical question and context:\n{x['input']}\n"
        "Answer carefully. /no_think"
    )
    prompt_msgs = [
        {"role":"system", "content":SYSTEM_PROMPT},
        {"role":"user", "content":user_text}
    ]
    full_msgs = prompt_msgs + [{"role":"assistant", "content":x["output"]}]
    p_ids = ids(chat_text(prompt_msgs, True))
    f_ids = ids(chat_text(full_msgs, False))

    # Find the common token prefix to mask prompt tokens robustly.
    common = 0
    while common < min(len(p_ids), len(f_ids)) and p_ids[common] == f_ids[common]:
        common += 1

    # Left-truncate long sequences so the answer is retained.
    removed = max(0, len(f_ids) - MAX_LENGTH)
    f_ids = f_ids[removed:]
    prompt_len = max(0, common - removed)
    labels = f_ids.copy()
    labels[:prompt_len] = [-100] * prompt_len
    valid = any(v != -100 for v in labels)
    return {"input_ids":f_ids, "attention_mask":[1]*len(f_ids), "labels":labels, "valid":valid}

sample = tokenize_example(train_dataset[0])
print("Sequence length:", len(sample["input_ids"]))
print("Answer tokens:", sum(v != -100 for v in sample["labels"]))
assert len(sample["input_ids"]) == len(sample["labels"])
assert sample["valid"], "نمونه آزمایشی پس از برش، توکن پاسخ ندارد."
print("Tokenization test passed.")

## 10. توکن‌سازی کل مجموعه

In [ ]:
tokenized_train = train_dataset.map(
    tokenize_example, remove_columns=train_dataset.column_names,
    desc="Tokenizing train"
)
tokenized_eval = eval_dataset.map(
    tokenize_example, remove_columns=eval_dataset.column_names,
    desc="Tokenizing validation"
)
tokenized_train = tokenized_train.filter(lambda row: row["valid"]).remove_columns(["valid"])
tokenized_eval = tokenized_eval.filter(lambda row: row["valid"]).remove_columns(["valid"])
print("Tokenized:", len(tokenized_train), len(tokenized_eval))
assert len(tokenized_train) > 0 and len(tokenized_eval) > 0

## 11. تنظیمات آموزش

In [ ]:
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True,
    pad_to_multiple_of=8, label_pad_token_id=-100, return_tensors="pt"
)
args_dict = dict(
    output_dir="/kaggle/working/MediGuideAI",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True, bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    seed=SEED
)
ta_sig = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in ta_sig:
    args_dict["eval_strategy"] = "epoch"
elif "evaluation_strategy" in ta_sig:
    args_dict["evaluation_strategy"] = "epoch"
training_args = TrainingArguments(**args_dict)
print("Training arguments ready.")

## 12. آموزش

In [ ]:
trainer_kwargs = dict(
    model=model, args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=collator
)
tr_sig = inspect.signature(Trainer.__init__).parameters
if "processing_class" in tr_sig:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer
trainer = Trainer(**trainer_kwargs)
train_result = trainer.train()
print(train_result)

## 13. ذخیره آداپتر

In [ ]:
adapter_dir = "/kaggle/working/MediGuideAI/adapter"
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("Saved:", adapter_dir)

## 14. ارزیابی loss

In [ ]:
metrics = trainer.evaluate()
print(json.dumps(metrics, indent=2, ensure_ascii=False))

## 15. تست پاسخ

In [ ]:
model.eval()
def ask_mediguide(question, context="", max_new_tokens=220):
    content = f"Question: {question}"
    if context.strip():
        content += f"\nScientific context: {context}"
    content += "\nGive an educational answer; do not diagnose or prescribe. /no_think"
    messages = [
        {"role":"system", "content":SYSTEM_PROMPT},
        {"role":"user", "content":content}
    ]
    prompt = chat_text(messages, True)
    batch = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    batch = {k:v.to(model.device) for k,v in batch.items()}
    with torch.no_grad():
        generated = model.generate(
            **batch, max_new_tokens=max_new_tokens, do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    answer_tokens = generated[0, batch["input_ids"].shape[1]:]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

print(ask_mediguide("What is the difference between a virus and a bacterium?"))

## 16. فشرده‌سازی برای دانلود

In [ ]:
import shutil
zip_path = shutil.make_archive(
    "/kaggle/working/MediGuideAI_adapter", "zip", adapter_dir
)
print("ZIP:", zip_path)

## یادداشت‌های مهم
- خروجی اصلی این آموزش آداپتر LoRA است، نه وزن کامل ادغام‌شده مدل.
- مدل روی GPU شماره 0 اجرا می‌شود؛ این کد از هر دو T4 برای تقسیم حافظه استفاده نمی‌کند.
- داده‌های این دو مجموعه عمدتاً انگلیسی هستند؛ توانایی فارسی تضمین نمی‌شود.
- یک epoch و loss پایین به معنی صحت پزشکی نیست. برای انتشار عمومی به آزمون مستقل، سنجش ایمنی، بررسی پزشک و داده فارسیِ دارای مجوز و بازبینی‌شده نیاز است.
- اگر هر سلولی خطا داد، متن کامل traceback و شماره سلول را بفرست؛ سازگاری کتابخانه‌ها و ساختار دیتاست‌ها ممکن است با گذر زمان تغییر کند.